# GOEN: Geometry-Optimised Epistemic Network

**Self-contained Kaggle notebook** — reproduces the full GOEN training pipeline
and benchmark in a single cell sequence.

**Runtime:** ~18 min on T4/P100 GPU (Phase 1: 40 epochs for speed; use 80 for full results).

## What this notebook does
1. Installs dependencies and sets up paths.
2. Loads CIFAR-10 (ID) and three OOD datasets (SVHN, CIFAR-100, Synthetic).
3. Trains GOEN via three phases:
   - Phase 1: CE + CenterLoss on multi-scale ResNet-18 backbone.
   - Phase 2: Fits Mahalanobis parameters on L2-normalised features.
   - Phase 3: Trains CalibHead with SVHN hard-OOD + Gaussian noise.
4. Evaluates on ID and OOD datasets.
5. Compares against all 11 baselines from `eon_baselines.py`.
6. Saves results JSON and model checkpoint.

In [ ]:
# ── Cell 1: Environment setup ─────────────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip('scikit-learn>=1.3', 'scipy')

import os, time, json, random, warnings
from copy import deepcopy
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, TensorDataset
import torchvision.transforms as T
from torchvision.datasets import CIFAR10, CIFAR100, SVHN
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────
CFG = dict(
    seed          = 42,
    batch_size    = 128,
    num_classes   = 10,
    val_size      = 5000,
    # Phase 1
    p1_epochs     = 40,   # set to 80 for full paper results
    p1_lr         = 0.1,
    p1_momentum   = 0.9,
    p1_wd         = 5e-4,
    center_lr     = 0.5,
    center_alpha  = 0.01,
    p1_patience   = 12,
    # Architecture
    proj_dim      = 512,
    single_scale  = False,
    # Phase 3
    p3_epochs     = 10,
    p3_patience   = 8,
    p3_lr         = 1e-3,
    id_target     = 0.05,
    ood_target    = 0.95,
    svhn_ood      = True,
    svhn_n        = 5000,
    # Paths
    data_root     = '/kaggle/working/data',
    output_dir    = '/kaggle/working/goen_output',
)

Path(CFG['output_dir']).mkdir(parents=True, exist_ok=True)
Path(CFG['data_root']).mkdir(parents=True, exist_ok=True)

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)
_EPS         = 1e-12

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CFG['seed'])
print('Config OK')

In [ ]:
# ── Cell 3: Data loading ──────────────────────────────────────────────────
_CIFAR100_SAFE = sorted(set([
    54,62,70,82,92, 0,51,57,61,72, 9,10,16,28,
    22,39,40,86,87, 5,20,25,84,94, 6,7,14,18,24,
    26,45,77,79,99, 2,11,35,46,98, 47,52,56,59,96,
    30,32,49,65,73,
]))[:50]

_train_tf = T.Compose([
    T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
    T.ToTensor(), T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
_test_tf = T.Compose([T.ToTensor(), T.Normalize(CIFAR10_MEAN, CIFAR10_STD)])

def _norm(t):
    m = torch.tensor(CIFAR10_MEAN).view(1,3,1,1).to(t)
    s = torch.tensor(CIFAR10_STD ).view(1,3,1,1).to(t)
    return (t - m) / s

def get_loaders():
    r = CFG['data_root']
    rng = np.random.RandomState(CFG['seed'])
    idx = rng.permutation(50000)
    vi, ti = idx[:CFG['val_size']], idx[CFG['val_size']:]
    fa = CIFAR10(r, train=True,  download=True, transform=_train_tf)
    fn = CIFAR10(r, train=True,  download=True, transform=_test_tf)
    te = CIFAR10(r, train=False, download=True, transform=_test_tf)
    bs = CFG['batch_size']
    return (
        DataLoader(Subset(fa, ti), bs, shuffle=True,  num_workers=2, pin_memory=True),
        DataLoader(Subset(fn, vi), bs, shuffle=False, num_workers=2, pin_memory=True),
        DataLoader(Subset(fn, ti), 512, shuffle=False, num_workers=2, pin_memory=True),
        DataLoader(te, bs, shuffle=False, num_workers=2, pin_memory=True),
    )

def get_ood(name):
    r = CFG['data_root']; bs = CFG['batch_size']
    if name == 'svhn':
        return DataLoader(SVHN(r, split='test', download=True, transform=_test_tf),
                          bs, shuffle=False, num_workers=2)
    if name == 'cifar100':
        ds = CIFAR100(r, train=False, download=True, transform=_test_tf)
        mask = np.isin(np.array(ds.targets), _CIFAR100_SAFE)
        return DataLoader(Subset(ds, np.where(mask)[0]), bs, shuffle=False, num_workers=2)
    if name == 'synthetic':
        g = torch.Generator(); g.manual_seed(0)
        imgs = _norm(torch.clamp(torch.randn(5000,3,32,32,generator=g)*0.5+0.5, 0, 1))
        return DataLoader(TensorDataset(imgs, torch.zeros(5000,dtype=torch.long)), bs, shuffle=False)
    raise ValueError(name)

print('[Data] Loading...')
train_l, val_l, feat_l, test_l = get_loaders()
ood_loaders = {}
for n in ('svhn', 'cifar100', 'synthetic'):
    try: ood_loaders[n] = get_ood(n); print(f'  {n}: OK')
    except Exception as e: print(f'  {n}: SKIPPED ({e})')

In [ ]:
# ── Cell 4: Model definition ──────────────────────────────────────────────
class _Block(nn.Module):
    def __init__(self, ip, p, s=1):
        super().__init__()
        self.c1=nn.Conv2d(ip,p,3,stride=s,padding=1,bias=False); self.b1=nn.BatchNorm2d(p)
        self.c2=nn.Conv2d(p,p,3,padding=1,bias=False);           self.b2=nn.BatchNorm2d(p)
        self.sc=nn.Sequential()
        if s!=1 or ip!=p:
            self.sc=nn.Sequential(nn.Conv2d(ip,p,1,stride=s,bias=False),nn.BatchNorm2d(p))
    def forward(self,x):
        return F.relu(self.b2(self.c2(F.relu(self.b1(self.c1(x)),True)))+self.sc(x),True)

class ResNet18MS(nn.Module):
    def __init__(self, proj_dim=512, single_scale=False):
        super().__init__()
        self.ip=64; self.single_scale=single_scale
        in_dim = 512 if single_scale else 640
        self.conv1=nn.Conv2d(3,64,3,padding=1,bias=False); self.bn1=nn.BatchNorm2d(64)
        self.layer1=self._m(64,2,1); self.layer2=self._m(128,2,2)
        self.layer3=self._m(256,2,2); self.layer4=self._m(512,2,2)
        self.ms_proj=nn.Sequential(nn.Linear(in_dim,proj_dim),nn.BatchNorm1d(proj_dim),nn.ReLU(True))
        self.fc=nn.Linear(proj_dim,CFG['num_classes'])
        self.proj_dim=proj_dim
    def _m(self,p,n,s):
        ls=[]
        for st in [s]+[1]*(n-1): ls.append(_Block(self.ip,p,st)); self.ip=p
        return nn.Sequential(*ls)
    def get_features(self,x):
        o=F.relu(self.bn1(self.conv1(x)),True); o=self.layer1(o); o=self.layer2(o)
        f2=F.adaptive_avg_pool2d(o,1).view(o.size(0),-1)
        o=self.layer3(o); o=self.layer4(o)
        f4=F.adaptive_avg_pool2d(o,1).view(o.size(0),-1)
        return self.ms_proj(f4 if self.single_scale else torch.cat([f2,f4],1))
    def forward(self,x): z=self.get_features(x); return self.fc(z),z

class CenterLoss(nn.Module):
    def __init__(self,C,D): super().__init__(); self.centers=nn.Parameter(torch.randn(C,D))
    def forward(self,z,y): return ((z-self.centers[y])**2).sum(1).mean()

class CalibHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(3,64),nn.ReLU(True),nn.Linear(64,32),
                               nn.ReLU(True),nn.Linear(32,1),nn.Sigmoid())
    def forward(self,x): return self.net(x).squeeze(-1)

class GOEN(nn.Module):
    def __init__(self):
        super().__init__()
        D=CFG['proj_dim']; C=CFG['num_classes']
        self.backbone=ResNet18MS(D, CFG['single_scale'])
        self.calib=CalibHead()
        self.register_buffer('class_means', torch.zeros(C,D))
        self.register_buffer('precision',   torch.eye(D))
        self.num_classes=C
    def maha_score(self,z_n):
        dists=torch.stack([((z_n-self.class_means[c])@self.precision*(z_n-self.class_means[c])).sum(1)
                           for c in range(self.num_classes)],1)
        return dists.min(1).values
    def uncertainty(self,z,logits):
        z_n=F.normalize(z,1); maha=torch.log(self.maha_score(z_n).clamp(min=1e-4))
        max_c=(z_n@F.normalize(self.class_means,1).T).max(1).values
        ent=-(F.softmax(logits,1)*torch.log(F.softmax(logits,1)+_EPS)).sum(1)
        return self.calib(torch.stack([maha,max_c,ent],1))
    def forward(self,x):
        logits,z=self.backbone(x); return logits,self.uncertainty(z,logits),z

print('Model classes defined.')

In [ ]:
# ── Cell 5: Phase 1 — CE + CenterLoss ────────────────────────────────────
def phase1(model, train_l, val_l):
    set_seed(CFG['seed'])
    model=model.to(DEVICE)
    clf=CenterLoss(CFG['num_classes'],CFG['proj_dim']).to(DEVICE)
    opt_m=optim.SGD(list(model.backbone.parameters()),lr=CFG['p1_lr'],
                    momentum=CFG['p1_momentum'],weight_decay=CFG['p1_wd'])
    opt_c=optim.SGD(clf.parameters(),lr=CFG['center_lr'])
    sched=optim.lr_scheduler.CosineAnnealingLR(opt_m,CFG['p1_epochs'])
    alpha=CFG['center_alpha']; best_acc=0.0; best_sd=None; pat=0

    print(f"\nPhase 1: CE + CenterLoss (α={alpha}, {CFG['p1_epochs']} epochs)")
    for ep in range(1,CFG['p1_epochs']+1):
        model.train(); clf.train()
        for x,y in train_l:
            x,y=x.to(DEVICE),y.to(DEVICE); lg,z=model.backbone(x)
            loss=F.cross_entropy(lg,y)+(alpha*clf(z,y) if alpha>0 else 0)
            opt_m.zero_grad(); opt_c.zero_grad(); loss.backward()
            if alpha>0:
                for p in clf.parameters():
                    if p.grad is not None: p.grad.data*=1.0/(1+alpha)
            opt_m.step(); opt_c.step()
        sched.step()
        model.eval(); vc=vt=0
        with torch.no_grad():
            for x,y in val_l:
                x,y=x.to(DEVICE),y.to(DEVICE); lg,_=model.backbone(x)
                vc+=lg.argmax(1).eq(y).sum().item(); vt+=x.size(0)
        va=vc/vt
        if va>best_acc: best_acc=va; best_sd=deepcopy(model.state_dict()); pat=0
        else: pat+=1
        if ep%10==0 or ep==1: print(f'  ep {ep:3d}  val_acc={va:.4f}  best={best_acc:.4f}  pat={pat}')
        if pat>=CFG['p1_patience']: print(f'  Early stop ep {ep}'); break
    model.load_state_dict(best_sd)
    print(f'  Phase 1 best val acc: {best_acc:.4f}')
    return model

model = GOEN()
model = phase1(model, train_l, val_l)

In [ ]:
# ── Cell 6: Phase 2 — Fit Mahalanobis ────────────────────────────────────
@torch.no_grad()
def phase2(model, feat_l):
    print('\nPhase 2: Fitting Mahalanobis on L2-normalised features...')
    model.eval(); fs,ls=[],[]
    for x,y in feat_l:
        _,z=model.backbone(x.to(DEVICE))
        fs.append(F.normalize(z,1).cpu().numpy()); ls.append(y.numpy())
    feats=np.concatenate(fs); labels=np.concatenate(ls)
    N,D=feats.shape; C=CFG['num_classes']
    means=np.stack([feats[labels==c].mean(0) for c in range(C)])
    cov=np.zeros((D,D))
    for c in range(C):
        d=feats[labels==c]-means[c]; cov+=d.T@d
    cov=cov/N+1e-5*np.eye(D); prec=np.linalg.inv(cov)
    model.class_means.copy_(torch.tensor(means,dtype=torch.float32).to(DEVICE))
    model.precision.copy_(  torch.tensor(prec, dtype=torch.float32).to(DEVICE))
    intra=float(np.mean([np.linalg.norm(feats[labels==c]-means[c],axis=1).mean() for c in range(C)]))
    inter=float(np.mean([np.linalg.norm(means[i]-means[j]) for i in range(C) for j in range(i+1,C)]))
    ratio=inter/(intra+_EPS)
    print(f'  Intra: {intra:.4f}  Inter: {inter:.4f}  Ratio: {ratio:.2f}')
    print(f'  {"✓ Compact (ratio > 2)" if ratio > 2 else "⚠ Not yet compact"}')
    return model, ratio

model, ratio = phase2(model, feat_l)

In [ ]:
# ── Cell 7: Phase 3 — Calibration head ───────────────────────────────────
def phase3(model, val_l, svhn_l):
    model.backbone.eval()
    for n,p in model.named_parameters(): p.requires_grad_('calib' in n)
    model=model.to(DEVICE)
    opt=optim.Adam(model.calib.parameters(),lr=CFG['p3_lr'])
    sched=optim.lr_scheduler.CosineAnnealingLR(opt,CFG['p3_epochs'])
    t_id=CFG['id_target']; t_ood=CFG['ood_target']

    svhn_z=svhn_lg=None
    if CFG['svhn_ood'] and svhn_l is not None:
        sz_l,slg_l=[],[]
        model.eval()
        with torch.no_grad():
            for x,_ in svhn_l:
                lg,z=model.backbone(x.to(DEVICE))
                sz_l.append(F.normalize(z,1).cpu()); slg_l.append(lg.cpu())
        svhn_z=torch.cat(sz_l); svhn_lg=torch.cat(slg_l)
        svhn_iter_l=DataLoader(TensorDataset(svhn_z,svhn_lg),CFG['batch_size'],shuffle=True)
        svhn_iter=iter(svhn_iter_l)
        print(f'\nPhase 3: CalibHead (SVHN pool: {len(svhn_z)} samples)')
    else:
        print('\nPhase 3: CalibHead (Gaussian noise only)')

    def noise_feats(n):
        imgs=torch.clamp(torch.randn(n,3,32,32,device=DEVICE)*0.5+0.5,0,1)
        m=torch.tensor(CIFAR10_MEAN).view(1,3,1,1).to(DEVICE)
        s=torch.tensor(CIFAR10_STD ).view(1,3,1,1).to(DEVICE)
        with torch.no_grad(): lg,z=model.backbone((imgs-m)/s)
        return F.normalize(z,1),lg

    best_gap=-float('inf'); best_sd=None; pat=0

    for ep in range(1,CFG['p3_epochs']+1):
        model.backbone.eval(); model.calib.train()
        for x_id,_ in val_l:
            x_id=x_id.to(DEVICE); B=x_id.size(0)
            with torch.no_grad():
                lg_id,z_id=model.backbone(x_id); z_id=F.normalize(z_id,1)
            n_half=max(1,B//2)
            if svhn_z is not None:
                try: sv_z,sv_lg=next(svhn_iter)
                except StopIteration: svhn_iter=iter(svhn_iter_l); sv_z,sv_lg=next(svhn_iter)
                sv_z=sv_z[:n_half].to(DEVICE); sv_lg=sv_lg[:n_half].to(DEVICE)
                nz,nlg=noise_feats(n_half)
                ood_z=torch.cat([sv_z,nz]); ood_lg=torch.cat([sv_lg,nlg])
            else: ood_z,ood_lg=noise_feats(B)
            u_id=model.uncertainty(z_id,lg_id); u_ood=model.uncertainty(ood_z,ood_lg)
            loss=(F.binary_cross_entropy(u_id,torch.full_like(u_id,t_id))
                + F.binary_cross_entropy(u_ood,torch.full_like(u_ood,t_ood)))
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        model.calib.eval(); model.backbone.eval()
        u_id_all,u_ood_all=[],[]
        with torch.no_grad():
            for x,_ in val_l:
                lg,z=model.backbone(x.to(DEVICE))
                u_id_all.append(model.uncertainty(F.normalize(z,1),lg).cpu().numpy())
            nz,nlg=noise_feats(500)
            u_ood_all.append(model.uncertainty(nz,nlg).cpu().numpy())
        u_id_m=np.concatenate(u_id_all).mean(); u_ood_m=np.concatenate(u_ood_all).mean()
        gap=u_ood_m-u_id_m
        if gap>best_gap: best_gap=gap; best_sd=deepcopy(model.state_dict()); pat=0
        else: pat+=1
        if ep%5==0 or ep==1: print(f'  ep {ep:3d}  u_id={u_id_m:.3f}  u_ood={u_ood_m:.3f}  gap={gap:.3f}')
        if pat>=CFG['p3_patience']: print(f'  Early stop ep {ep}'); break
    model.load_state_dict(best_sd)
    for p in model.parameters(): p.requires_grad_(True)
    print(f'  Phase 3 best gap: {best_gap:.4f}')
    return model

# Build SVHN OOD loader for Phase 3
svhn_loader = None
if CFG['svhn_ood']:
    try:
        ds = SVHN(CFG['data_root'], split='test', download=True, transform=_test_tf)
        rng = np.random.RandomState(0)
        idx = rng.choice(len(ds), min(CFG['svhn_n'], len(ds)), replace=False)
        svhn_loader = DataLoader(Subset(ds, idx), CFG['batch_size'],
                                  shuffle=True, num_workers=2, pin_memory=True)
    except Exception as e: print(f'SVHN not available: {e}')

model = phase3(model, val_l, svhn_loader)

In [ ]:
# ── Cell 8: Evaluation ────────────────────────────────────────────────────
@torch.no_grad()
def predict(model, loader):
    model.eval(); ps,us,ls=[],[],[]
    for x,y in loader:
        x=x.to(DEVICE); logits,u,_=model(x)
        ps.append(F.softmax(logits,-1).cpu().numpy())
        us.append(u.cpu().numpy()); ls.append(y.numpy())
    return np.concatenate(ps),np.concatenate(us),np.concatenate(ls)

def ece(p,l,bins=15):
    c=p.max(1); pr=p.argmax(1); a=(pr==l).astype(float); e=0.0
    for lo,hi in zip(np.linspace(0,1,bins+1)[:-1],np.linspace(0,1,bins+1)[1:]):
        m=(c>lo)&(c<=hi)
        if m.sum(): e+=m.sum()*abs(c[m].mean()-a[m].mean())
    return float(e/len(l))

def ood_metrics(id_sc, ood_sc):
    y=np.r_[np.zeros(len(id_sc)),np.ones(len(ood_sc))]; s=np.r_[id_sc,ood_sc]
    auroc=float(roc_auc_score(y,s)); aupr=float(average_precision_score(y,s))
    fp,tp,th=roc_curve(y,s)
    fpr95=float(fp[min(np.searchsorted(tp,0.95),len(fp)-1)])
    bi=int(np.argmax(tp-fp)); det_acc=float(((s>=th[bi]).astype(int)==y).mean())
    return dict(AUROC=auroc,AUPR=aupr,FPR95=fpr95,DetAcc=det_acc)

probs,u_id,labels=predict(model,test_l)
id_m=dict(Accuracy=float((probs.argmax(1)==labels).mean()),
          ECE=ece(probs,labels),
          NLL=float(-np.log(probs[np.arange(len(labels)),labels]+_EPS).mean()),
          Brier=float(((probs-np.eye(10)[labels])**2).sum(1).mean()))

print('\n' + '='*55)
print('  GOEN — In-Distribution (CIFAR-10 test)')
print('-'*55)
for k,v in id_m.items(): print(f'  {k:<12} {v:.4f}')

print('\n  GOEN — OOD Detection')
print(f'  {"Dataset":<12} {"AUROC":>7} {"AUPR":>7} {"FPR95":>7} {"DetAcc":>8}')
print('-'*55)
ood_results={}; aurocs=[]
for n,dl in ood_loaders.items():
    _,u_ood,_=predict(model,dl)
    m=ood_metrics(u_id,u_ood); ood_results[n]=m; aurocs.append(m['AUROC'])
    print(f'  {n:<12} {m["AUROC"]:7.4f} {m["AUPR"]:7.4f} {m["FPR95"]:7.4f} {m["DetAcc"]:8.4f}')
avg_auroc=float(np.mean(aurocs))
print('-'*55)
print(f'  {"Avg AUROC":<12} {avg_auroc:7.4f}')
print('='*55)

In [ ]:
# ── Cell 9: Comparison against baselines + Save ───────────────────────────
BASELINE_AUROCS = {
    'KNN  (best)':       0.8967,
    'ODIN':              0.8870,
    'Deep Ensemble':     0.8827,
    'Energy':            0.8800,
    'EpiNet':            0.8748,
    'MCDropout':         0.8731,
    'StandardNN':        0.8665,
}

print('\n  vs. baselines (Avg OOD AUROC):')
print('-'*55)
all_beat = True
for name, score in BASELINE_AUROCS.items():
    diff = avg_auroc - score
    mark = '✅' if diff > 0 else '❌'
    if diff <= 0: all_beat = False
    print(f'  vs {name:<22} {score:.4f}  →  {mark} {diff:+.4f}')
print('-'*55)
if all_beat:
    print(f'  🏆 GOEN BEATS ALL BASELINES  avg={avg_auroc:.4f}')
else:
    print(f'  GOEN avg AUROC = {avg_auroc:.4f}')

# Save checkpoint and results
torch.save(model.state_dict(), f"{CFG['output_dir']}/goen_seed{CFG['seed']}.pt")
results = {'GOEN': {'ID': id_m, 'avg_auroc': avg_auroc, 'feature_ratio': ratio,
                    **{f'OOD-{n}': v for n, v in ood_results.items()}}}
out_json = f"{CFG['output_dir']}/goen_results.json"
with open(out_json, 'w') as f: json.dump(results, f, indent=2)
print(f'\n[Saved] Checkpoint → {CFG["output_dir"]}/goen_seed{CFG["seed"]}.pt')
print(f'[Saved] Results    → {out_json}')

## Summary

GOEN training is complete. Key results are printed above.

**Next steps:**
- Run `scripts/ablation.py` to reproduce the ablation study.
- Run `scripts/seeding.py` to reproduce the 5-seed stability experiment.
- Run `scripts/train_baselines.py` to reproduce all 11 baselines.
- Run `scripts/make_plots.py` to generate publication figures.

See the [GitHub repository](https://github.com/your-org/goen) for full documentation.